# TinyRPSNet

In [46]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

In [71]:
height = 32
width = 32

transform = transforms.Compose([ # Resize and greyscale
    transforms.Resize((height, width)),
	transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
])

train_dataset = datasets.ImageFolder("data/train", transform=transform)
test_dataset = datasets.ImageFolder("data/train_val", transform=transform)

batch_size = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4
)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4
)

In [73]:
print(f"Number of instances: \n Training: {len(train_loader.dataset)}, Testing: {len(test_loader.dataset)}\n")
print(f"Number of batches: \n Training: {len(train_loader)}, Testing: {len(test_loader)}")

Number of instances: 
 Training: 2172, Testing: 545

Number of batches: 
 Training: 136, Testing: 35


In [74]:
class TinyRPSNet(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()

        # Initial conv
        self.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=8,
            kernel_size=3,
            stride=2,
            padding=1
        )

        # Depthwise + Pointwise block 1
        self.dw1 = nn.Conv2d(
            in_channels=8,
            out_channels=8,
            kernel_size=3,
            padding=1,
            groups=8  # depthwise
        )
        self.pw1 = nn.Conv2d(
            in_channels=8,
            out_channels=16,
            kernel_size=1
        )

        # Depthwise + Pointwise block 2
        self.dw2 = nn.Conv2d(
            in_channels=16,
            out_channels=16,
            kernel_size=3,
            padding=1,
            groups=16
        )
        self.pw2 = nn.Conv2d(
            in_channels=16,
            out_channels=32,
            kernel_size=1
        )

        # Global average pooling + classifier
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(32, num_classes)

    def forward(self, x):
        # x: (B, 1, 64, 64)

        x = F.leaky_relu(self.conv1(x), negative_slope=0.1)
        x = F.leaky_relu(self.pw1(self.dw1(x)), negative_slope=0.1)
        x = F.leaky_relu(self.pw2(self.dw2(x)), negative_slope=0.1)

        x = self.pool(x)           # (B, 32, 1, 1)
        x = torch.flatten(x, 1)    # (B, 32)
        x = self.fc(x)             # (B, num_classes)

        return x  # logits (no softmax here)

Sanity check

In [75]:
def init_weights(m):
    if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

model = TinyRPSNet(num_classes=3)
model.apply(init_weights)
dummy_input = torch.randn(1, 1, 64, 64)
logits = model(dummy_input)
logits.shape

torch.Size([1, 3])

# Instantiating the model

In [76]:
device = "cpu"
print(f"Using {device} device")
model = TinyRPSNet().to(device)

print(model)

Using cpu device
TinyRPSNet(
  (conv1): Conv2d(1, 8, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (dw1): Conv2d(8, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=8)
  (pw1): Conv2d(8, 16, kernel_size=(1, 1), stride=(1, 1))
  (dw2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16)
  (pw2): Conv2d(16, 32, kernel_size=(1, 1), stride=(1, 1))
  (pool): AdaptiveAvgPool2d(output_size=1)
  (fc): Linear(in_features=32, out_features=3, bias=True)
)


In [77]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [53]:
def train(data_loader: DataLoader):
	model.train()
	size = len(data_loader.dataset)
	for batch, (X, y) in enumerate(data_loader):
		X, y = X.to(device), y.to(device)

		#Prediction error
		pred = model(X)
		loss = loss_fn(pred, y)

		#Backpropagation
		loss.backward()
		optimizer.step()
		optimizer.zero_grad()

		if batch % 50 == 0:
			loss, current = loss.item(), (batch + 1) * len(X)
			print(f"loss: {loss:>7f} [{current:>5d}/{size:>5d}]")

In [54]:
def test(data_loader: DataLoader):
	model.eval()
	size = len(data_loader.dataset)
	num_batches = len(data_loader)
	test_loss, correct = 0, 0
	
	with torch.no_grad():
		for X, y in data_loader:
			X, y = X.to(device), y.to(device)
			pred = model(X)
			loss = loss_fn(pred, y)
			test_loss += loss.item()
			correct += (pred.argmax(1) == y).type(torch.float).sum().item()
			
	test_loss /= num_batches
	correct /= size
	print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")


In [79]:
epochs = 20
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_loader)
    test(test_loader)
print("Done!")

Epoch 1
-------------------------------
loss: 1.070498 [   16/ 2172]
loss: 1.102317 [  816/ 2172]
loss: 1.099647 [ 1616/ 2172]
Test Error: 
 Accuracy: 33.4%, Avg loss: 1.100712 

Epoch 2
-------------------------------
loss: 1.087031 [   16/ 2172]
loss: 1.077663 [  816/ 2172]
loss: 1.107770 [ 1616/ 2172]
Test Error: 
 Accuracy: 33.2%, Avg loss: 1.098324 

Epoch 3
-------------------------------
loss: 1.098689 [   16/ 2172]
loss: 1.105431 [  816/ 2172]
loss: 1.098024 [ 1616/ 2172]
Test Error: 
 Accuracy: 33.4%, Avg loss: 1.098874 

Epoch 4
-------------------------------
loss: 1.097891 [   16/ 2172]
loss: 1.100369 [  816/ 2172]
loss: 1.101136 [ 1616/ 2172]
Test Error: 
 Accuracy: 33.2%, Avg loss: 1.098377 

Epoch 5
-------------------------------
loss: 1.099407 [   16/ 2172]
loss: 1.098508 [  816/ 2172]
loss: 1.101173 [ 1616/ 2172]
Test Error: 
 Accuracy: 33.4%, Avg loss: 1.098915 

Epoch 6
-------------------------------
loss: 1.097353 [   16/ 2172]
loss: 1.097065 [  816/ 2172]
loss: 1

In [78]:
model.eval()

with torch.no_grad():
    x_batch, y_batch = next(iter(test_loader))
    out = model(x_batch)
    print(out[:5])

tensor([[-0.1346, -0.0250, -0.2066],
        [-0.1345, -0.0250, -0.2066],
        [-0.1346, -0.0250, -0.2066],
        [-0.1345, -0.0250, -0.2066],
        [-0.1346, -0.0250, -0.2066]])
